In [1]:
import random
import json
import os
from pathlib import Path
from typing import Dict, List, Any
import pandas as pd
import asyncio
import time

import os
from openai import AsyncOpenAI 
from agents import Agent, Runner 
from agents import set_default_openai_client
from agents import OpenAIResponsesModel 
from agents import set_tracing_disabled
set_tracing_disabled(True)#for jupyter, remove before moving to CLI
from pandas import read_csv
import copy 

import numpy as np
import joblib
import pickle


### Playbook definition

In [2]:
class Playbook:
    def __init__(self, jsonl_path: str, docs_root: str = "docs/playbook"):
        self.jsonl_path = Path(jsonl_path)
        self.docs_root = Path(docs_root)

        # --- Load & parse JSONL --------------------------------------------
        self.chunks: List[Dict[str, Any]] = []

        with open(self.jsonl_path, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue  # skip blank lines
                try:
                    chunk = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(
                        f"Malformed JSON on line {line_no} of {self.jsonl_path}"
                    ) from exc
                self.chunks.append(chunk)

        # Build quick look‑ups
        self.by_id: Dict[str, Dict[str, Any]] = {c["chunk_id"]: c for c in self.chunks}
        self.tag_index: Dict[str, List[str]] = {}
        for c in self.chunks:
            for tag in c["tags"]:
                self.tag_index.setdefault(tag, []).append(c["chunk_id"])

    # ------------------------------------------------------------
    #  API helpers
    # ------------------------------------------------------------
    def get_chunks_by_tags(self, tags: List[str]) -> List[Dict[str, Any]]:
        """Return the intersection of chunks that contain *all* supplied tags."""
        if not tags:
            return []

        # Start with the set for the first tag
        common = set(self.tag_index.get(tags[0], []))
        for t in tags[1:]:
            common &= set(self.tag_index.get(t, []))

        return [self.by_id[cid] for cid in common]

    def load_markdown_for_chunk(self, chunk: Dict[str, Any]) -> str:
        """
        The `doc_id` field points at the markdown file name *without* extension.
        e.g.  `doc_id="medical_necessity.md"` → load `docs/playbook/medical_necessity.md`
        """
        doc_file = self.docs_root / f"{chunk['doc_id']}"
        if not doc_file.exists():
            return f"[ERROR: missing {doc_file}]"

        return doc_file.read_text(encoding="utf-8")

    def load_markdown_for_tags(self, tags: List[str]) -> Dict[str, str]:
        """Return a mapping of doc_id → file contents for all chunks matching the tags."""
        chunks = self.get_chunks_by_tags(tags)
        return {c["doc_id"]: self.load_markdown_for_chunk(c) for c in chunks}


# claim_playbook = Playbook('data/playbook_chunks.jsonl')

In [3]:
# ───────────────────────────────────────────────────────────────────────
#  Imports
# ───────────────────────────────────────────────────────────────────────
from datetime import datetime

# Agents‑SDK core objects
from agents.agent import Agent,ToolContext  
from agents import SQLiteSession
from agents import FunctionTool

# The Responses‑only model provider
from agents import OpenAIResponsesModel
from pydantic import BaseModel

In [4]:
#todos
    #get session working
    #replace random selection in predict_denial_taxonomy with something else. I think its supposed to be a classifier of some sort
    #somehow handle unknown and missing values

In [5]:
import json
from typing import Any
from agents.agent_output import AgentOutputSchemaBase

class CustomDenialSchema(AgentOutputSchemaBase):
    def __init__(self):
        # Load your schema file
        with open("docs/workup_output_schema.json", "r") as f:
            self._schema = json.load(f)
    
    def is_plain_text(self) -> bool:
        return False  # We want JSON output, not plain text
    
    def is_strict_json_schema(self) -> bool:
        return False  # Your schema has advanced features (null unions, etc.) not in strict mode
    
    def name(self) -> str:
        return "DenialWorkupOutput"  # Human-readable name for the output type
    
    def json_schema(self) -> dict[str, Any]:
        return self._schema  # Returns your exact raw schema
    
    def validate_json(self, json_str: str) -> dict[str, Any]:
        try:
            result = json.loads(json_str)
            # Basic validation: check required top-level keys from your schema
            required = {"claim_id", "denial_taxonomy", "payment_analysis", "pursuit_recommendation", 
                       "reasons", "missing_fields", "open_questions", "recommended_next_steps", 
                       "supporting_playbook_citations", "draft_narrative", "trace"}
            if not all(key in result for key in required):
                raise ValueError("Missing required fields")
            return result  # Return as dict (no further typing)
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON: {e}")  # Use ValueError
        except Exception as e:
            raise ValueError(f"Validation failed: {e}")

In [19]:
class OllamaResponsesAgent():
    def __init__(
        self,
        *,
        ollama_base_url: str = "http://localhost:11434/v1",
        model_name: str = 'gpt-oss:20b',
        sqlite_path: str = "agentic.db",
    ):

        
        """
        Initialise the agent, its memory store, and the Ollama Responses model.

        Parameters
        ----------
        ollama_base_url : str
            Base URL of the Ollama instance 
        model_name : str
            Name of the model to use inside Ollama.
        sqlite_uri : str
            URI for the SQLite session store.
        """
        self.session = SQLiteSession(
            session_id = f"session_at_{time.strftime('%m_%d_%Y_%H_%M')}",
            db_path = sqlite_path
        )

        client = AsyncOpenAI(
            api_key='ollama',
            base_url=ollama_base_url,  # key detail: route requests to Ollama (local or cloud)
        )
        
        set_default_openai_client(client)

        self.model = OpenAIResponsesModel(
            model=model_name,
            openai_client=client,
        )
        
        
        pipeline = joblib.load('models/denial_classifier.pkl')  # Or 'output/denial_classifier.pkl'
        le = joblib.load('models/label_encoder.pkl')
        self.denial_taxonomy_classifier = {'pipeline':pipeline,'encoder':le,}
        
        self.tools = [
            self._predict_denial_taxonomy_tool(),
            self._retrieve_playbook_tool(),
        ]

        self.agent = Agent(
            name = "Claim Agent",
            model=self.model,
            tools=self.tools,
            output_type=CustomDenialSchema(),
            instructions="""
                        You are a patient advocate. 
                        Always call the tools predict_denial_taxonomy and retrieve_playbook.
                        If you're given a claim information dictionary, then your job is to suggest a recommendation for how to best proceed.
                        This recommendation can be one of the following options and nothing else: pursue, do_not_pursue, or needs_info. Always use
                        the retrieve_playbook tool to get more information and instructions specific to the type of claim and denial code.

                        Alternatively, you may be asked questions about a claim. Answer these questions with text, not conforming to the output schema.
                        """
        )
        self.playbook = Playbook('data/playbook_chunks.jsonl')
        self.tag_agent = Agent(
                name="Tag Assigning Agent",
                instructions="""
                You will be given an insurance claim and asked to assign any number of tags to it.
                If you find any of the following in the claim, it MUST be one of your tags: CO-16, CO-27, CO-29, CO-45, CO-50, CO-97
                Also assign any of the following tags if it appropriate: coding_bundling, eligibility, general, medical_necessity, missing_info, other, timely_filing, underpayment
                """,
                model=self.model,
            )


        self.question_agent = Agent(
            name = "Question Agent",
            model=self.model,
            instructions="""
                        You will be asked about a claim within this session. Answer using the history as context.
                        """
        )
        with open('docs/workup_output_schema.json','r') as f:
            output_schema = json.load(f)
        self.output_schema = output_schema
    
    def _predict_denial_taxonomy_tool(self) -> FunctionTool: 
        """Return a FunctionTool that gives the current UTC time."""
        # Load model and encoder
        def predict_with_explain(pipeline, le, sample_df):
            if sample_df['denial_text'].isna().all() and sample_df['denial_code'].isna().all():
                return {'label': 'unknown', 'confidence': 0.0, 'top_features': []}
            
            X_trans = pipeline.named_steps['preprocessor'].transform(sample_df)
            probs = pipeline.named_steps['classifier'].predict_proba(X_trans)
            pred_idx = np.argmax(probs, axis=1)[0]
            confidence = np.max(probs[0])
            
            if confidence < 0.5:#replace with entropy?
                return {'label': 'unknown', 'confidence': confidence, 'top_features': []}
            
            pred_label = le.inverse_transform([pred_idx])[0]
            
            # Top features
            coefs = pipeline.named_steps['classifier'].coef_[pred_idx]
            feature_names = (pipeline.named_steps['preprocessor'].named_transformers_['text'].get_feature_names_out().tolist() +
                             list(pipeline.named_steps['preprocessor'].named_transformers_['code'].get_feature_names_out()))
            top_indices = np.argsort(np.abs(coefs))[-10:]
            top_features = [(feature_names[i], float(coefs[i])) for i in top_indices[::-1]]
            
            return {'label': pred_label, 'confidence': float(confidence), 'top_features': top_features}
            
        async def predict_denial_taxonomy(_:ToolContext, denial_info: dict) -> str:
            """Predict denial for claim"""
            
            
            #type error handling
            if isinstance(denial_info,str):
                denial_info = json.loads(denial_info)
            if 'denial_info' in denial_info.keys():
                denial_code = denial_info['denial_info']
            else:
                denial_code = "UNKNOWN"
            if 'denial_text' in denial_info.keys():
                denial_text = denial_info['denial_text']
            else:
                denial_text = "UNKNOWN"

            #format claim for inference
            denied_claim = pd.DataFrame({
                'denial_code': [denial_code],
                'denial_text': [denial_text]
            })
            result = predict_with_explain(self.denial_taxonomy_classifier['pipeline'], self.denial_taxonomy_classifier['encoder'], denied_claim)
            return result #dict with confidence, prediction, and top features

        # The FunctionTool registers the function name, description, etc.
        return FunctionTool(
            name="predict_denial_taxonomy",
            description="predict denial reason for a claim. Provide denial code and denial text and no other parameters",
            on_invoke_tool=predict_denial_taxonomy,
            params_json_schema={
                    "type": "object",
                    "properties": {
                        "denial_code": {"type": "string", "description": "the 4-6 digit code representing the reason the claim was denied"}, 
                        "denial_text": {"type": "string", "description": "the text describing why the claim was denied"}, 
                    },
                    # "required": ["denial_code","denial_text"],
                },
        )
    def _retrieve_playbook_tool(self) -> FunctionTool: #todo: error because model gave 2 input params instead of 1. Apply to prompt###############################################################
        """Get additional instructions and context from playbook using denial code and denial reason"""

        async def retrieve_playbook(_:ToolContext,tag_prompt: str) -> str:
            """Get additional instructions and context from playbook using denial code and denial reason"""
            tags = await Runner.run(self.tag_agent,tag_prompt) 
            return self.playbook.load_markdown_for_tags(tags.final_output)

        # The FunctionTool registers the function name, description, etc.
        return FunctionTool(
            name="retrieve_playbook",
            description="Get additional instructions and context from playbook using denial code and denial reason. Provide a single string describing the claim and nothing else.",
            on_invoke_tool=retrieve_playbook,
            params_json_schema={
                    "type": "object",
                    "properties": {
                        # "tool_context": {"type": "string", "description": "Context for tool use"}, #specifying what claim info could help here
                        "claim_information": {"type": "string", "description": "the denial text for the claim"}, #specifying what claim info could help here
                    },
                    "required": ["claim_information"],
                    "additionalProperties": False,
                },
        )
    def _format_output(self) -> FunctionTool: #todo: error because model gave 2 input params instead of 1. Apply to prompt###############################################################
        """Get additional instructions and context from playbook using denial code and denial reason"""

        async def conform_to_output_schema(_:ToolContext,args) -> str:
            """Get additional instructions and context from playbook using denial code and denial reason"""
            tags = await Runner.run(self.tag_agent,tag_prompt) 
            return self.playbook.load_markdown_for_tags(tags.final_output)

        # The FunctionTool registers the function name, description, etc.
        return FunctionTool(
            name="retrieve_playbook",
            description="Get additional instructions and context from playbook using denial code and denial reason. Provide a single string describing the claim and nothing else.",
            on_invoke_tool=retrieve_playbook,
            params_json_schema={
                    "type": "object",
                    "properties": {
                        # "tool_context": {"type": "string", "description": "Context for tool use"}, #specifying what claim info could help here
                        "claim_information": {"type": "string", "description": "the denial text for the claim"}, #specifying what claim info could help here
                    },
                    "required": ["claim_information"],
                    "additionalProperties": False,
                },
        )
    # ------------------------------------------------------------------
    #  Public API: run a single turn
    # ------------------------------------------------------------------
    async def run(self, prompt: str, ask = False) -> str:
        """
        Send a prompt to the Agent and receive its response.

        The response may include tool calls; the Agent handles executing
        those tools automatically (via the FunctionTool infrastructure).

        Parameters
        ----------
        prompt : str
            The user message to send to the Agent.

        Returns
        -------
        str
            The raw text response from the Agent (after any tool calls).
        """
        if ask:
            result = await Runner.run(
                self.question_agent,
                prompt,
                session = self.session
            )
        else:
            result = await Runner.run(
                self.agent,
                prompt,
                session=self.session)

        return result

agent = OllamaResponsesAgent()
claims = read_csv('data/claims.csv')
results = []
rand_idx=random.randint(0,len(claims))
for i,row in claims.iterrows():
    if i < rand_idx or i > rand_idx + 3:
        continue
    try:
        claim_dict = row.to_dict()
        result = await agent.run(str(claim_dict))
        print('Claim ',i,'\n',result.final_output)
        results.append(result)
    except Exception as e:#todo: try again if failure
        print(e)
            

C:\Users\ryan\AppData\Local\Temp\ipykernel_62764\3503571901.py:241: RuntimeWarning: coroutine 'Runner.run' was never awaited
  result = await agent.run(str(claim_dict))


Claim  56 
 {'claim_id': 'C-000057', 'denial_taxonomy': {'category': 'needs_info', 'confidence': 0.34}, 'payment_analysis': {'billed_amount': 3917.94, 'paid_amount': 0.0, 'coverage_ratio': 0.0, 'co45_contract_paid_gate': True}, 'pursuit_recommendation': 'needs_info', 'reasons': ['Incomplete data, need eligibility verification', 'Follow up with payer for missing eligibility documentation', 'Once eligibility verified, submit appeal or resubmit claim'], 'missing_fields': ['eligibility verification data'], 'open_questions': ['What eligibility documents are required?', 'What is the process for obtaining eligibility verification?'], 'recommended_next_steps': [{'step': 'Collect eligibility verification', 'detail': 'Contact provider to obtain member eligibility confirmation.', 'citations': ['retrieve_playbook response']}, {'step': 'Resubmit claim after verification', 'detail': 'Once eligibility is verified, resubmit the claim with complete data.', 'citations': ['retrieve_playbook response']}],

In [20]:
print(len(results))

1


In [23]:
result = await agent.run('Tell me a cute story with information about the first claim from this session?',ask=True)
result.final_output

'**The Tale of the Little Claim and the Helpful HealthPlan Demo**\n\nOnce upon a time in the bustling town of Claimville, there lived a bright little claim named **C‑000057**. C‑000057 was a single‑digit hero, standing proud with its little number‑tall arms holding the famous CPT code **27447** (the “hip‑replacement helper”). It wore a shiny invoice sheet that read:\n\n> *Billed Amount:* **$3,917.94**  \n> *Paid Amount:* **$0.00**  \n> *Plan:* **ERISA**  \n> *Payer:* **HealthPlan Demo**\n\nC‑000057 loved to help patients feel better, but one day it found itself in a sticky situation. When the big, friendly office at HealthPlan Demo tried to process it, they blinked their eyes and said, “**CO‑16** – Rejected as unprocessable due to incomplete/invalid data. Provide eligibility verification.” Oh no! The little claim’s heart fluttered like a tiny drum.\n\nBut C‑000057 was not one to give up. It had heard whispers from the old paperwork library that the secret to a happy claim’s life was th

In [22]:
result = await agent.run('What about ther first one?',ask=True)
result.final_output

'The first claim you provided was **C‑000057**.'

In [ ]:
#todo:
    #somehow assess json format error
    #create another agent: one for ask and one for doing the claim workup
        #add this to tradoffs, but also imo the best way to do it because of how hard it sticks to the output schema
    #add tool to search the internet for claim denial policy, make a case for it to be approved.
        #there must be specific language detailing what is covered and what isnt under certain plans